In [12]:
import glob
import os
import re
import sys

import numpy as np
import optuna
import pandas as pd

In [13]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

from drGAT.metrics import get_parsed_df

In [14]:
def extract_method_data(filename):
    match = re.match(r"([A-Za-zA-Z0-9]+)_([a-z0-9]+)\.sqlite3", filename)
    return match.groups() if match else ("unknown", "unknown")

In [15]:
all_dfs = []

for file in glob.glob("*.sqlite3"):
    method, data = extract_method_data(os.path.basename(file))
    try:
        study = optuna.load_study(study_name=method, storage=f"sqlite:///{file}")
        df_all = study.trials_dataframe()

        # ✅ COMPLETE 状態の数を表示
        n_complete = (df_all["state"] == "COMPLETE").sum()
        print(f"✅ {file}: {n_complete} trials completed")

        df_valid = df_all.dropna(
            subset=["values_0", "values_1", "values_2", "values_3"]
        )

        if df_valid.shape[0] == 0:
            continue

        # user_attrs から評価指標を取り出す
        df_metrics = df_valid.iloc[:, 20:-2]
        df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
        parsed_df = get_parsed_df(df_metrics)
        parsed_df["n_complete"] = n_complete

        # ハイパーパラメータ列だけ抽出
        df_params = df_valid[
            [c for c in df_valid.columns if "params" in c]
        ].reset_index(drop=True)
        parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

        # method / data 列を追加
        parsed_df["method"] = method
        parsed_df["data"] = data

        all_dfs.append(parsed_df)

    except Exception as e:
        print(f"❌ Failed to parse {file}: {e}")

if all_dfs:
    summary_df = pd.concat(all_dfs, ignore_index=True)
    summary_df["AUPR_num"] = summary_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

    # 最良インデックス取得
    best_idxs = summary_df.groupby(["data", "method"])["AUPR_num"].idxmax()
    best_df = summary_df.loc[best_idxs].drop(columns=["AUPR_num"])

    # ✅ インデックス設定: method, data の順で MultiIndex
    best_df.set_index(
        [
            "data",
            "method",
        ],
        inplace=True,
    )

    # ✅ data をキーに並べ替え（index の level 1 をソート）
    best_df.sort_index(level="data", inplace=True)

    # 任意の順に並べたいカラム
    desired_order = ["n_complete", "ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]

    # その順に並び替える（残りのカラムはそのまま後ろへ）
    other_cols = [col for col in best_df.columns if col not in desired_order]
    best_df = best_df[desired_order + other_cols]

    # インデックスの data レベルを並び替え
    desired_data_order = [
        "nci",
        "gdsc1",
        "gdsc2",
        "ctrp",
    ]  # 小文字なら小文字にそろえる！

    # sort_index ではなく reindex で順序を明示
    best_df = best_df.reindex(
        best_df.index.reindex(
            sorted(best_df.index, key=lambda x: desired_data_order.index(x[0]))
        )[0]
    )

    display(best_df)
else:
    print("No valid studies found.")

✅ GAT_ctrp.sqlite3: 14 trials completed
✅ GAT_gdsc1.sqlite3: 15 trials completed
✅ GAT_gdsc2.sqlite3: 19 trials completed
✅ GAT_nci.sqlite3: 44 trials completed
✅ Transformer_gdsc1.sqlite3: 15 trials completed
✅ Transformer_ctrp.sqlite3: 20 trials completed
✅ Transformer_nci.sqlite3: 55 trials completed
✅ Transformer_gdsc2.sqlite3: 19 trials completed
✅ GATv2_gdsc1.sqlite3: 19 trials completed
✅ GATv2_ctrp.sqlite3: 12 trials completed
✅ GATv2_gdsc2.sqlite3: 19 trials completed
✅ GATv2_nci.sqlite3: 26 trials completed


n_complete              ACC        Precision  \
data  method                                                      
nci   GAT                  44  0.854 (± 0.008)  0.862 (± 0.012)   
      GATv2                26  0.848 (± 0.015)  0.853 (± 0.027)   
      Transformer          55  0.842 (± 0.011)  0.861 (± 0.012)   
gdsc1 GAT                  15  0.738 (± 0.007)  0.735 (± 0.013)   
      GATv2                19  0.742 (± 0.006)  0.734 (± 0.011)   
      Transformer          15  0.742 (± 0.006)  0.736 (± 0.013)   
gdsc2 GAT                  19  0.795 (± 0.014)  0.804 (± 0.015)   
      GATv2                19  0.792 (± 0.014)  0.802 (± 0.021)   
      Transformer          19  0.795 (± 0.014)  0.809 (± 0.025)   
ctrp  GAT                  14  0.835 (± 0.005)  0.851 (± 0.013)   
      GATv2                12  0.839 (± 0.006)  0.857 (± 0.006)   
      Transformer          20   0.84 (± 0.005)  0.852 (± 0.007)   

                            Recall               F1            AUROC  \
data  method                                                           
nci   GAT          0.837 (± 0.024)   0.849 (± 0.01)  0.913 (± 0.005)   
      GATv2         0.837 (± 0.02)  0.844 (± 0.012)   0.91 (± 0.009)   
      Transformer  0.811 (± 0.017)  0.835 (± 0.011)  0.905 (± 0.006)   
gdsc1 GAT          0.724 (± 0.017)   0.73 (± 0.008)  0.815 (± 0.006)   
      GATv2         0.738 (± 0.01)  0.736 (± 0.008)  0.818 (± 0.005)   
      Transformer  0.733 (± 0.012)  0.735 (± 0.009)  0.818 (± 0.004)   
gdsc2 GAT           0.798 (± 0.02)  0.801 (± 0.016)  0.863 (± 0.012)   
      GATv2        0.796 (± 0.018)  0.799 (± 0.014)  0.863 (± 0.012)   
      Transformer   0.791 (± 0.01)    0.8 (± 0.013)  0.865 (± 0.012)   
ctrp  GAT          0.829 (± 0.024)   0.84 (± 0.007)  0.908 (± 0.004)   
      GATv2        0.829 (± 0.015)  0.843 (± 0.006)  0.912 (± 0.003)   
      Transformer  0.838 (± 0.012)  0.845 (± 0.005)  0.912 (± 0.003)   

                              AUPR     Balanced_ACC  params_T_max  \
data  method                                                        
nci   GAT          0.901 (± 0.004)  0.854 (± 0.008)        1228.0   
      GATv2          0.9 (± 0.008)  0.848 (± 0.015)           NaN   
      Transformer  0.901 (± 0.008)  0.842 (± 0.011)        2167.0   
gdsc1 GAT          0.809 (± 0.007)  0.738 (± 0.007)           NaN   
      GATv2         0.81 (± 0.006)  0.742 (± 0.006)        2066.0   
      Transformer  0.811 (± 0.006)  0.742 (± 0.006)        1119.0   
gdsc2 GAT          0.859 (± 0.012)  0.794 (± 0.014)           NaN   
      GATv2        0.859 (± 0.015)  0.792 (± 0.014)        1322.0   
      Transformer  0.862 (± 0.017)  0.795 (± 0.015)           NaN   
ctrp  GAT          0.913 (± 0.006)  0.836 (± 0.004)        2806.0   
      GATv2        0.917 (± 0.004)  0.839 (± 0.005)           NaN   
      Transformer  0.916 (± 0.004)   0.84 (± 0.005)        1409.0   

                  params_activation  ...  params_heads  params_hidden1  \
data  method                         ...                                 
nci   GAT                      relu  ...           8.0           631.0   
      GATv2                    relu  ...           5.0           464.0   
      Transformer              gelu  ...           2.0           683.0   
gdsc1 GAT                      gelu  ...           5.0           554.0   
      GATv2                    relu  ...           6.0           387.0   
      Transformer              gelu  ...           6.0           757.0   
gdsc2 GAT                      gelu  ...           3.0           419.0   
      GATv2                    relu  ...           6.0           425.0   
      Transformer              gelu  ...           4.0           444.0   
ctrp  GAT                      relu  ...           4.0           957.0   
      GATv2                    gelu  ...           4.0           604.0   
      Transformer              relu  ...           4.0           988.0   

                   params_hidden2  params_hidden3  params_is_zero_p

In [16]:
best_df[desired_order]

n_complete              ACC        Precision  \
data  method                                                      
nci   GAT                  44  0.854 (± 0.008)  0.862 (± 0.012)   
      GATv2                26  0.848 (± 0.015)  0.853 (± 0.027)   
      Transformer          55  0.842 (± 0.011)  0.861 (± 0.012)   
gdsc1 GAT                  15  0.738 (± 0.007)  0.735 (± 0.013)   
      GATv2                19  0.742 (± 0.006)  0.734 (± 0.011)   
      Transformer          15  0.742 (± 0.006)  0.736 (± 0.013)   
gdsc2 GAT                  19  0.795 (± 0.014)  0.804 (± 0.015)   
      GATv2                19  0.792 (± 0.014)  0.802 (± 0.021)   
      Transformer          19  0.795 (± 0.014)  0.809 (± 0.025)   
ctrp  GAT                  14  0.835 (± 0.005)  0.851 (± 0.013)   
      GATv2                12  0.839 (± 0.006)  0.857 (± 0.006)   
      Transformer          20   0.84 (± 0.005)  0.852 (± 0.007)   

                            Recall               F1            AUROC  \
data  method                                                           
nci   GAT          0.837 (± 0.024)   0.849 (± 0.01)  0.913 (± 0.005)   
      GATv2         0.837 (± 0.02)  0.844 (± 0.012)   0.91 (± 0.009)   
      Transformer  0.811 (± 0.017)  0.835 (± 0.011)  0.905 (± 0.006)   
gdsc1 GAT          0.724 (± 0.017)   0.73 (± 0.008)  0.815 (± 0.006)   
      GATv2         0.738 (± 0.01)  0.736 (± 0.008)  0.818 (± 0.005)   
      Transformer  0.733 (± 0.012)  0.735 (± 0.009)  0.818 (± 0.004)   
gdsc2 GAT           0.798 (± 0.02)  0.801 (± 0.016)  0.863 (± 0.012)   
      GATv2        0.796 (± 0.018)  0.799 (± 0.014)  0.863 (± 0.012)   
      Transformer   0.791 (± 0.01)    0.8 (± 0.013)  0.865 (± 0.012)   
ctrp  GAT          0.829 (± 0.024)   0.84 (± 0.007)  0.908 (± 0.004)   
      GATv2        0.829 (± 0.015)  0.843 (± 0.006)  0.912 (± 0.003)   
      Transformer  0.838 (± 0.012)  0.845 (± 0.005)  0.912 (± 0.003)   

                              AUPR  
data  method                        
nci   GAT          0.901 (± 0.004)  
      GATv2          0.9 (± 0.008)  
      Transformer  0.901 (± 0.008)  
gdsc1 GAT          0.809 (± 0.007)  
      GATv2         0.81 (± 0.006)  
      Transformer  0.811 (± 0.006)  
gdsc2 GAT          0.859 (± 0.012)  
      GATv2        0.859 (± 0.015)  
      Transformer  0.862 (± 0.017)  
ctrp  GAT          0.913 (± 0.006)  
      GATv2        0.917 (± 0.004)  
      Transformer  0.916 (± 0.004)